In [1]:
import os
import pickle
import ijson
import jsonlines
import pandas as pd
import numpy as np
import json
import copy
from tqdm import tqdm
from collections import defaultdict

In [2]:
try:
    META_FILE = "../../output/grocery/item2attributes.json"   # 相对 notebook 的新路径
    data = json.load(open(META_FILE, "r", encoding="utf-8"))
    print(f"Loaded metadata for {len(data)} items from {META_FILE}")
except FileNotFoundError:
    print(f"Error: {META_FILE} not found. Did grocery_data_process.py run successfully?")
    exit()
except json.JSONDecodeError:
    print(f"Error: {META_FILE} is not a valid JSON file.")
    exit()

Loaded metadata for 15101 items from ../../output/grocery/item2attributes.json


In [3]:
# %% [code]

# --- 配置 ---
META_FILE_PATH = "../../output/grocery/item2attributes.json"   # ← 新路径
NUM_SAMPLES_TO_VIEW = 3 

actual_format = "unknown"

print(f"--- 检查文件格式: {META_FILE_PATH} ---")
try:
    with open(META_FILE_PATH, "rb") as f:  # 使用二进制模式读取以检查开头字符
        start_bytes = f.read(5).strip()
        if start_bytes.startswith(b"{"):
            actual_format = "json_dict"
            print("文件为 JSON 字典")
        elif start_bytes.startswith(b"["):
            actual_format = "json_array"
            print("文件为 JSON 数组")
        else:
            # 可能是 JSONL 或其他格式，尝试按行读取
            f.seek(0)
            with open(META_FILE_PATH, "r", encoding="utf-8") as f_text:
                for i, line in enumerate(f_text):
                    line = line.strip()
                    if not line:
                        continue
                    if line.startswith("{") and line.endswith("}"):
                        try:
                            json.loads(line)
                            actual_format = "jsonl"
                            print("文件为 JSONL 格式")
                            break
                        except json.JSONDecodeError:
                            pass
                    if i > 5:
                        break
                if actual_format == "unknown":
                    print(
                        f"错误：无法识别文件格式。文件开头: {start_bytes.decode(errors='ignore')}..."
                    )

    # --- 抽样查看 ---
    if actual_format == "json_dict":
        print(f"\n--- 抽样查看前 {NUM_SAMPLES_TO_VIEW} 个物品 (使用 ijson) ---")
        items_viewed = 0
        try:
            with open(META_FILE_PATH, "rb") as f:
                items_stream = ijson.kvitems(f, "")
                for i, (key, value) in enumerate(items_stream):
                    if i < NUM_SAMPLES_TO_VIEW:
                        print(f"\nItem Key (asin): {key}")
                        print(
                            f"Value (完整): \n{json.dumps(value, indent=2, ensure_ascii=False)}"
                        )
                    else:
                        break
                    items_viewed += 1
            if items_viewed == 0:
                print(
                    "未能从文件中解析出任何键值对。请检查文件是否为空或格式是否为 {key: value, ...}。"
                )
        except Exception as e:
            print(" item2attributes.json 可能并非 JSON 字典，且顶级键是物品ID。")

    elif actual_format == "jsonl":
        # ... (JSONL 的抽样逻辑不变) ...
        print(f"\n--- 抽样查看前 {NUM_SAMPLES_TO_VIEW} 行 (JSONL 格式) ---")
        try:
            with open(META_FILE_PATH, "r", encoding="utf-8") as f:
                for i, line in enumerate(f):
                    line = line.strip()
                    if not line:
                        continue
                    if i < NUM_SAMPLES_TO_VIEW:
                        try:
                            obj = json.loads(line)
                            print(f"\nLine {i+1}:")
                            print(
                                f"  Content (完整): \n{json.dumps(obj, indent=2, ensure_ascii=False)}"
                            )
                        except json.JSONDecodeError:
                            print(f"Line {i+1} 不是有效的 JSON: {line[:100]}...")
                    else:
                        break
        except Exception as e:
            print(f"读取 JSONL 文件时出错: {e}")

    elif actual_format == "unknown":
        print("\n无法进行抽样，因为文件格式未知。请手动检查文件。")

except FileNotFoundError:
    print(f"错误: 找不到文件 {META_FILE_PATH}")
    actual_format = "not_found"
except Exception as e:
    print(f"打开或读取文件时发生未知错误: {e}")
    actual_format = "error"

--- 检查文件格式: ../../output/grocery/item2attributes.json ---
文件为 JSON 字典

--- 抽样查看前 3 个物品 (使用 ijson) ---

Item Key (asin): 4639725043
Value (完整): 
{
  "category": [
    "Grocery & Gourmet Food",
    "Beverages",
    "Coffee, Tea & Cocoa",
    "Tea",
    "Black"
  ],
  "tech1": "",
  "description": [
    "Lipton Yellow Label Tea use only the finest tea leaves.  These leaves are specially cut to expose more of the juices, then they are curled into tiny beads to seal in more of the flavor.  Enjoy a hot cup of Lipton Yellow Label Tea today.",
    "Tea",
    "Statements regarding dietary supplements have not been evaluated by the FDA and are not intended to diagnose, treat, cure, or prevent any disease or health condition."
  ],
  "fit": "",
  "title": "Lipton Yellow Label Tea (loose tea) - 450g",
  "also_buy": [
    "B00886E4K0",
    "B00CREXSHY",
    "B001QTRGAQ",
    "B002EYZM4O",
    "B000JSQK70",
    "B00FMTETUQ",
    "B001VIIXXQ",
    "B002UP153Y",
    "B07DZ4M75Z",
    "B00N48M0OO",
   

In [4]:
example_dict = {}
for item_dict in tqdm(data.values()):
    example_dict.update(item_dict)
example_dict.keys()


  0%|                                                                                                                                        | 0/15101 [00:00<?, ?it/s]


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15101/15101 [00:00<00:00, 365945.34it/s]

dict_keys(['category', 'tech1', 'description', 'fit', 'title', 'also_buy', 'tech2', 'brand', 'feature', 'rank', 'also_view', 'main_cat', 'similar_item', 'date', 'price', 'asin', 'imageURL', 'imageURLHighRes', 'details'])

Grocery 数据结构说明：

去掉不需要的属性（如 region_id, url），剩下的属性可以分为文本类和列表类

文本类：直接添加到prompt即可

列表类：先把列表中的element组成文本，再添加到prompt

Grocery 保留字段：
- asin: 商品ID (唯一标识符)
- title: review标题
- description: 商品描述
- categories: 分类（字符串格式）
- brand: 商品品牌
- rank: 商品评分
- date: 日期
- price: 价格

舍弃字段：
- imageURL & imageURLHighRes: 何意味
- similar_item：污染
- main_cat: 冗余
- fit & also_buy & also_view
- details: 无用

非空保留：
- tech1 & tech2 & feature & description

In [ ]:
instruction = "The grocery or gourmet food item has the following attributes: \n"

In [6]:
item_data = {}
for item_dict in tqdm(data.values()):
    item_prompt = copy.deepcopy(instruction)
    item_id = None
    for key, value in item_dict.items():
        if key in ["fit", "also_buy", "also_view", "similar_item", "imageURL", "imageURLHighRes", "main_cat", "details"]:   # drop longitude and latitude
            continue
        elif key in ["asin"]:  # get the item id
            item_id = value
        elif key in ["description", "feature", "tech1", "tech2"]:    # list type attributes
            attri_str = ""
            for meta_str in value:
                attri_str += (meta_str + ", ")
            if len(value) == 0:
                attri_str = "none, "
            attri_str = attri_str.replace("\n", " ").replace(";", ".")
            if len(attri_str) > 150:
                attri_str = attri_str[:150]
            attri_prompt = key + " is " + attri_str[:-2] + "; "    # [:-2] is to remove the last ", "
            item_prompt += attri_prompt
        elif key in ["price", "date"]:    # list type attributes
            attri_str = ""
            if len(value) == 0:
                attri_str = "unkown, "
            else:
                attri_str = value
            if len(attri_str) > 50:
                attri_str = attri_str[:50]
            attri_prompt = key + " is " + attri_str[:-2] + "; "    # [:-2] is to remove the last ", "
            item_prompt += attri_prompt
        else:   # str type attributes 
            if len(value) > 150:
                value = value[:150]
            if key in ["rank"]:
                attri_prompt = key + " is " + str(value).replace("\n", " ").replace(";", ".").replace(" (", "") + "; "
            else:
                attri_prompt = key + " is " + str(value).replace("\n", " ").replace(";", ".") + "; "
            item_prompt += attri_prompt
    if item_id:
        item_data[item_id] = item_prompt[:-2]
    else:
        raise ValueError("No item id")


  0%|                                                                                                                                        | 0/15101 [00:00<?, ?it/s]


 13%|████████████████▏                                                                                                         | 2005/15101 [00:00<00:00, 20044.00it/s]


 27%|████████████████████████████████▍                                                                                         | 4010/15101 [00:00<00:00, 19261.98it/s]


 39%|███████████████████████████████████████████████▉                                                                          | 5939/15101 [00:00<00:00, 19008.48it/s]


 54%|█████████████████████████████████████████████████████████████████▎                                                        | 8082/15101 [00:00<00:00, 19940.31it/s]


 67%|████████████████████████████████████████████████████████████████████████████████▊                                        | 10079/15101 [00:00<00:00, 19666.14it/s]


 80%|████████████████████████████████████████████████████████████████████████████████████████████████▌                        | 12048/15101 [00:00<00:00, 19298.16it/s]


 93%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌        | 14046/15101 [00:00<00:00, 19513.57it/s]


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15101/15101 [00:00<00:00, 19471.91it/s]

In [7]:
json.dump(item_data, open("../../output/grocery/item_str.json", "w"))

In [8]:
# 显示一个示例
print("Sample generated prompt:")
sample_key = list(item_data.keys())[76]
print(f"Item ID: {sample_key}")
print(f"Prompt: {item_data[sample_key]}")
print(f"\nTotal items processed: {len(item_data)}")

Sample generated prompt:
Item ID: B0000GHNW2
Prompt: The grocery or gourmet food item has the following attributes: 
category is ['Grocery & Gourmet Food', 'Sauces, Gravies & Marinades', 'Hot Sauce']; tech1 is none; description is Valentina Salsa Picante Mexican Sauce, Statements regarding dietary supplements have not been evaluated by the FDA and are not intended to diagnose,; title is Valentina Salsa Picante Mexican Sauce; tech2 is none; brand is ValentinA; feature is none; rank is 9,469 in Grocery & Gourmet Food; date is unkown; price is $5.

Total items processed: 15101


In [ ]:
import os

OUTPUT_DIR = "../../output/grocery/"              # 统一输出目录

# convert to jsonline
def save_data(data_path, data):
    """write all_data list to a new jsonl"""
    outfile = os.path.join(OUTPUT_DIR, data_path)
    with jsonlines.open(outfile, "w") as w:
        for meta_data in data:
            w.write(meta_data)

# 读取 id_map
id_map_path = os.path.join(OUTPUT_DIR, "id_map.json")
id_map = json.load(open(id_map_path, "r"))["item2id"]

# 组装 json_data
json_data = []
missing = 0
for key, value in item_data.items():
    if key not in id_map:            # ← 关键判断
        missing += 1
        continue                     # 跳过已被删除的 item
    json_data.append({
        "input": value,
        "target": "",
        "item": key,
        "item_id": id_map[key]
    })

print(f"Skipped {missing} items without id_map entry")
save_data("item_str.jsonline", json_data)
print("✅ Successfully saved item_str.jsonline")

Skipped 9 items without id_map entry
